# Human 다음 행동·간격 공동 기준선
왜: 지금까지는 시간만 예측했습니다. 이번에는 직전 행동을 알면 **다음 시도와 간격**의 분포를 더 잘 예측하는지 확인합니다.

여러 사람의 전반적인 기록 성향을 학습하는 **부분 기준선**입니다. 보드 판단·배치 선택·Hold·개인 복제·실제 게임 제어를 학습한 모델이 아닙니다. 휴리스틱으로 배치를 대신 정하는 계획도 아닙니다.

[실험 명세](../../../ai-harness/ml-experiments/experiments/human-action-timing-joint-baseline.md) · [전체 계획](../../../docs/ai-ml/strategy/ai-plan.md)

실행 순서: 공통 정의 → 작은 예제/내부 검사 → 가상 Smoke → 선택적인 Full → 결과 해석.
**기본 모두 실행은 실제 dataset/Test를 열지 않습니다.** Python 테스트나 shell 명령을 먼저 실행할 필요가 없습니다.


In [ ]:
import hashlib
import json
import math
import os
import platform
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import confusion_matrix, f1_score, precision_recall_fscore_support

EXPERIMENT_ID = "human-action-timing-joint-baseline"
ROOT = next(p.resolve() for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "ai-harness").is_dir() and (p / "package.json").is_file())
SOURCE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_action_timing_joint_baseline.ipynb"
BASELINE_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_timing_baseline.ipynb"
READINESS_NOTEBOOK = ROOT / "scripts/ml/notebooks/human_action_data_readiness.ipynb"


def definition_cells(path, ids=None):
    """실행 셀·사용자 output은 제외하고 승인한 정의 셀만 가져온다."""
    notebook = json.loads(path.read_text(encoding="utf-8"))
    selected = [{"id": c["id"], "source": "".join(c["source"])} for c in notebook["cells"]
                if c["cell_type"] == "code" and
                (c["id"] in ids if ids is not None else "definitions" in c["metadata"].get("tags", []))]
    encoded = (json.dumps(selected, ensure_ascii=False, sort_keys=True,
                          allow_nan=False, indent=2) + "\n").encode("utf-8")
    return selected, hashlib.sha256(encoded).hexdigest()


base_cells, BASELINE_DEFS_SHA = definition_cells(
    BASELINE_NOTEBOOK, {"shared-definitions", "input-definitions", "metric-definitions"})
_, READINESS_DEFS_SHA = definition_cells(READINESS_NOTEBOOK)
assert len(base_cells) == 3 and BASELINE_DEFS_SHA == "47ecd252a0143d9a716a0a24327afde2003ad8927586fc9bd197f9a4881c218b"
assert READINESS_DEFS_SHA == "a14916cdd4ef6cae2557e239ad60bf710d287a7475073706a7d619be752d14df"
baseline = {"__name__": "joint_shared"}
for cell in base_cells:
    exec(compile(cell["source"], str(BASELINE_NOTEBOOK), "exec"), baseline)
baseline["shared"]["EXPERIMENT_ID"] = EXPERIMENT_ID
for name in ("require", "finite", "integer", "json_bytes", "sha256_file", "load_inputs",
             "validate_sample", "checked_paths", "publish_report",
             "prepare_distribution", "crps", "distribution_summary"):
    globals()[name] = baseline[name]

ACTIONS = ("hard_drop", "move_left", "move_right", "rotate", "rotate_ccw", "soft_drop")
VARIANTS = ("joint_global", "joint_previous_action")
EXPECTED_FILES = {
    "train": (149, 41151, "c42c01d6fba0eb7128b77c0773a1669a9be620530a2fb99171dcca67ff3fb7c3"),
    "validation": (31, 10180, "7e9099916aae3e0f7c3875d8169e88f7847b0f9f517a506c78b2776d4d44660f"),
}


## 1. 입력과 정답을 나누기

| 역할 | 값 |
| --- | --- |
| 예측 입력 x | 이미 기록된 직전 `previousAction` |
| 행동 정답 y | `nextActionDiagnostic` |
| 시간 정답 t | `recordedIntervalMs`: 처리 시도 사이 간격(ms) |
| 검증/집계 | game·player·piece·sourceLine·인접 input index |

정답 y와 t는 예측 함수에 전달하지 않습니다. 첫 행동 대기·Hold는 이 dataset에 없으며, 0ms도 보존합니다.
Game 수가 많은 사람이 더 “옳은 사람”이라는 뜻이 아닙니다. Train은 pair 단위, 평가는 게임별 동등 비중으로 해석합니다.


In [ ]:
def validate_joint_rows(rows, split):
    """기존 schema/lineage 검사에 공동 과제의 label·정수 간격·중복 검사를 추가한다."""
    require(isinstance(rows, list) and rows, "EMPTY_ROWS")
    games = {r.get("gameId") for r in rows if isinstance(r, dict)}
    seen, positions = set(), {}
    for row in rows:
        validate_sample(row, split, games)
        require(row["previousAction"] in ACTIONS and row["nextActionDiagnostic"] in ACTIONS,
                "JOINT_ACTION_INVALID")
        require(type(row["recordedIntervalMs"]) is int and row["recordedIntervalMs"] >= 0,
                "JOINT_INTERVAL_INVALID")
        require(row["sampleId"] not in seen, "DUPLICATE_SAMPLE")
        seen.add(row["sampleId"])
        group = (row["gameId"], row["playerNumber"], row["pieceIndex"], row["sourceLine"])
        for index, action in ((row["previousInputIndex"], row["previousAction"]),
                              (row["currentInputIndex"], row["nextActionDiagnostic"])):
            key = (*group, index)
            require(key not in positions or positions[key] == action, "ACTION_LINEAGE_CONFLICT")
            positions[key] = action
    return rows


def sample_row(split, game, index, previous, following, interval, source_line=1):
    """가상 예제를 실제와 같은 row 계약으로 만들며 외부 파일은 읽지 않는다."""
    lineage = {"gameId": game, "playerNumber": 1, "pieceIndex": 0, "sourceLine": source_line,
               "previousInputIndex": index, "currentInputIndex": index + 1}
    return {"schemaVersion": 1, "sampleId": hashlib.sha256(json_bytes(lineage)).hexdigest(),
            **lineage, "previousAction": previous, "nextActionDiagnostic": following,
            "recordedIntervalMs": interval, "split": split}


def synthetic_rows():
    rows = {"train": [], "validation": []}
    for split, games in (("train", 3), ("validation", 2)):
        for g in range(games):
            sequence = ("move_left", "rotate", "move_left", "hard_drop")
            for i in range(3):
                rows[split].append(sample_row(split, f"synthetic-{split}-{g}", i,
                    sequence[i], sequence[i + 1], (0, 100, 300)[(i + g) % 3]))
    return rows


## 2. 비교할 두 기준선

- **joint_global**: 직전 행동을 보지 않고 다음 행동 확률과 행동별 시간 분포를 계산합니다.
- **joint_previous_action**: 직전 행동별로 다음 행동 확률과 시간 분포를 계산합니다.

행동 확률은 count + Laplace alpha=1입니다. 시간은 실제 정수 ms의 경험 분포입니다.
미학습 직전 행동 → global, 빈 (직전, 다음) 조합 → 다음 행동별 시간, 그 행동도 없으면 → 전체 시간으로 fallback합니다.

두 모델 모두 count 기반입니다. 신경망·optimizer·scaler·개인 식별 feature는 없습니다.


In [ ]:
def fit_joint(train):
    """Train만 집계한다. 정렬된 count/CDF 표본이 실험용 bundle의 단일 원천이다."""
    validate_joint_rows(train, "train")
    ordered = sorted(train, key=lambda r: r["sampleId"])

    def cell(rows):
        return {"counts": [sum(r["nextActionDiagnostic"] == y for r in rows) for y in ACTIONS],
                "times": {y: sorted(r["recordedIntervalMs"] for r in rows
                                    if r["nextActionDiagnostic"] == y) for y in ACTIONS}}

    return {"vocabulary": list(ACTIONS), "alpha": 1, "fitWeighting": "pair",
            "global": cell(ordered),
            "byPrevious": {x: cell([r for r in ordered if r["previousAction"] == x])
                           for x in ACTIONS if any(r["previousAction"] == x for r in ordered)},
            "allTimes": sorted(r["recordedIntervalMs"] for r in ordered)}


def forecast(bundle, variant, previous_action):
    """x만 받는다. 미래 action/time이나 Human 최종 배치를 입력으로 쓰지 않는다."""
    require(variant in VARIANTS and previous_action in ACTIONS, "FORECAST_INPUT_INVALID")
    local = variant == "joint_previous_action" and previous_action in bundle["byPrevious"]
    cell = bundle["byPrevious"][previous_action] if local else bundle["global"]
    counts = np.asarray(cell["counts"], dtype=float)
    probability = (counts + 1) / (counts.sum() + len(ACTIONS))
    values, sources = [], []
    for y in ACTIONS:
        if cell["times"][y]:
            values.append(cell["times"][y])
            sources.append("xy" if local else "y")
        elif bundle["global"]["times"][y]:
            values.append(bundle["global"]["times"][y])
            sources.append("y")
        else:
            values.append(bundle["allTimes"])
            sources.append("all")
    require(np.isfinite(probability).all() and (probability > 0).all()
            and abs(float(probability.sum()) - 1) < 1e-12, "PROBABILITY_INVALID")
    return probability, values, sources, variant == "joint_previous_action" and not local


## 3. 행동 확률과 시간 분포를 평가하기

**NLL (낮을수록 좋음)**: 실제 다음 행동에 얼마나 높은 확률을 주었는가? 단위는 natural log입니다.

**F1 (높을수록 좋음)**: 가장 높은 확률의 행동 하나를 고른 결과입니다. 6개 행동을 고정하고 없는 class도 0으로 계산하므로 행동별 support를 함께 봅니다.

**Marginal CRPS (ms, 낮을수록 좋음)**: 다음 행동을 모르는 상태에서, 예측 행동 확률로 시간 분포들을 섞어 평가합니다.
정답 행동을 알려준 conditional CRPS는 **oracle 진단**이며 실제 시간 guardrail과 다릅니다.

NLL과 CRPS는 단위가 달라 하나의 점수로 더하지 않습니다.


In [ ]:
def prepare_mixture(probabilities, values):
    """가중 경험 분포를 한 번 정렬하고 누적합으로 정확한 CRPS를 준비한다."""
    require(len(probabilities) == len(values) and len(values) > 0, "MIXTURE_SHAPE_INVALID")
    require(all(finite(float(p)) and p > 0 for p in probabilities)
            and abs(math.fsum(probabilities) - 1) < 1e-12, "MIXTURE_PROBABILITY_INVALID")
    for v in values:
        prepare_distribution(v)
    x = np.concatenate([np.asarray(v, dtype=float) for v in values])
    w = np.concatenate([np.full(len(v), float(p) / len(v)) for p, v in zip(probabilities, values)])
    order = np.argsort(x, kind="stable")
    x, w = x[order], w[order]
    w /= math.fsum(w)
    cw = np.cumsum(w)
    prefix_w = np.concatenate(([0.0], cw))
    prefix_xw = np.concatenate(([0.0], np.cumsum(x * w)))
    half = math.fsum(float(a * b * (2 * c - b - 1)) for a, b, c in zip(x, w, cw))
    require(finite(half) and np.isfinite(prefix_xw).all(), "MIXTURE_OVERFLOW")
    return x, w, prefix_w, prefix_xw, half


def mixture_crps(prepared, observations):
    """행마다 거리 행렬을 만들지 않고 E|X-t| - E|X-X'|/2를 계산한다."""
    require(len(observations) > 0 and all(finite(t) and t >= 0 for t in observations),
            "OBSERVATION_INVALID")
    x, w, pw, px, half = prepared
    y = np.asarray(observations, dtype=float)
    k = np.searchsorted(x, y, side="right")
    result = y * pw[k] - px[k] + px[-1] - px[k] - y * (pw[-1] - pw[k]) - half
    require(np.isfinite(result).all() and (result >= -1e-8).all(), "MIXTURE_CRPS_INVALID")
    return np.maximum(result, 0)


def score_summary(actual, predicted, nll, timing, oracle, median_error):
    return {"pairs": len(actual), "nextActionNll": float(np.mean(nll)),
            "macroF1": float(f1_score(actual, predicted, labels=list(range(6)),
                                      average="macro", zero_division=0)),
            "marginalCrpsMs": float(np.mean(timing)),
            "oracleConditionalCrpsMs": float(np.mean(oracle)),
            "mixtureMedianMaeMs": float(np.mean(median_error))}


def evaluate_joint(bundle, rows):
    """같은 Validation 표본을 평가한다. 예측 함수에는 x만 제공하고 cache를 재사용한다."""
    validate_joint_rows(rows, "validation")
    rows = sorted(rows, key=lambda r: r["sampleId"])
    actual = np.array([ACTIONS.index(r["nextActionDiagnostic"]) for r in rows])
    times = np.array([r["recordedIntervalMs"] for r in rows])
    result = {}
    for variant in VARIANTS:
        predicted = np.zeros(len(rows), dtype=int)
        nll, timing, oracle, errors = (np.zeros(len(rows)) for _ in range(4))
        forecasts, fallback = {}, Counter()
        fallback_mass = 0.0
        for x in ACTIONS:
            index = np.array([i for i, r in enumerate(rows) if r["previousAction"] == x], dtype=int)
            if not len(index):
                continue
            probability, values, sources, unseen = forecast(bundle, variant, x)
            mixture = prepare_mixture(probability, values)
            summary = distribution_summary(mixture[0], mixture[1])
            predicted[index] = int(np.argmax(probability))
            nll[index] = -np.log(probability[actual[index]])
            timing[index] = mixture_crps(mixture, times[index].tolist())
            errors[index] = abs(times[index] - summary["medianMs"])
            for yi, source in enumerate(sources):
                subset = index[actual[index] == yi]
                if len(subset):
                    oracle[subset] = crps(prepare_distribution(values[yi]), times[subset].tolist())
                fallback[source] += len(index)
                if source != ("xy" if variant == "joint_previous_action" and not unseen else "y"):
                    fallback_mass += float(probability[yi]) * len(index)
            fallback["unseenPreviousRows"] += int(unseen) * len(index)
            forecasts[x] = {"actionProbabilities": probability.tolist(), "timeSources": sources,
                            "timeSupport": [len(v) for v in values], "mixtureTime": summary,
                            "validationPairs": len(index)}
        game_rows = []
        for game in sorted({r["gameId"] for r in rows}):
            idx = np.array([i for i, r in enumerate(rows) if r["gameId"] == game])
            game_rows.append({"gameId": game, **score_summary(actual[idx], predicted[idx],
                nll[idx], timing[idx], oracle[idx], errors[idx])})
        metric_keys = ("nextActionNll", "macroF1", "marginalCrpsMs",
                       "oracleConditionalCrpsMs", "mixtureMedianMaeMs")
        precision, recall, f1, support = precision_recall_fscore_support(
            actual, predicted, labels=list(range(6)), zero_division=0)
        majority = int(np.argmax(bundle["global"]["counts"]))
        result[variant] = {
            "gameMacro": {k: float(np.mean([g[k] for g in game_rows])) for k in metric_keys},
            "pairMicro": score_summary(actual, predicted, nll, timing, oracle, errors),
            "perGame": game_rows,
            "perAction": [{"action": a, "trueSupport": int(support[i]),
                           "predictedSupport": int(np.sum(predicted == i)),
                           "precision": float(precision[i]), "recall": float(recall[i]), "f1": float(f1[i])}
                          for i, a in enumerate(ACTIONS)],
            "confusionMatrix": confusion_matrix(actual, predicted, labels=list(range(6))).tolist(),
            "forecastByPrevious": forecasts,
            "fallbackCalls": dict(sorted(fallback.items())),
            "fallbackCallDenominator": len(rows) * len(ACTIONS),
            "fallbackProbabilityMassMean": fallback_mass / len(rows),
            "alwaysTrainMajorityReference": {"action": ACTIONS[majority],
                "gameMacroF1": float(np.mean([f1_score(
                    actual[[i for i, r in enumerate(rows) if r["gameId"] == g["gameId"]]],
                    np.full(g["pairs"], majority), labels=list(range(6)), average="macro", zero_division=0)
                    for g in game_rows]))}}
    a, b = (result[v]["gameMacro"] for v in VARIANTS)
    delta = {k: b[k] - a[k] for k in a}
    status = ("conditional_signal_observed" if delta["nextActionNll"] < -1e-12
              and delta["marginalCrpsMs"] <= 1e-9 and delta["macroF1"] >= -1e-12
              else "tradeoff_requires_review" if delta["nextActionNll"] < -1e-12 else "no_clear_gain")
    per_game_delta = [{"gameId": a["gameId"], **{k: b[k] - a[k] for k in delta}}
                      for a, b in zip(result[VARIANTS[0]]["perGame"], result[VARIANTS[1]]["perGame"])]
    nll_delta = [r["nextActionNll"] for r in per_game_delta]
    return {"models": result, "gameMacroDelta": delta, "perGameDelta": per_game_delta,
            "analysisStatus": status,
            "gameNllDeltaSummary": {"median": float(np.median(nll_delta)),
                "worst": float(max(nll_delta)), "improved": sum(d < -1e-12 for d in nll_delta),
                "same": sum(abs(d) <= 1e-12 for d in nll_delta),
                "worse": sum(d > 1e-12 for d in nll_delta)}}


## 4. 작은 예제로 손계산 확인하기

설명용으로 행동을 두 개로 축약합니다. 실제 모델은 여섯 개입니다.

`left → rotate 100ms, left → rotate 200ms, left → drop 300ms, right → drop 100ms`

alpha=1이면 전체 rotate/drop 확률은 0.5/0.5, left 조건은 0.6/0.4입니다.
left의 시간 mixture는 100ms:0.3, 200ms:0.3, 300ms:0.4입니다. 정답이 rotate인지 몰라도 이 분포를 만들 수 있습니다.


In [ ]:
worked = prepare_mixture([0.6, 0.4], [[100, 200], [300]])
assert np.allclose(worked[1], [0.3, 0.3, 0.4])
assert np.allclose(mixture_crps(prepare_mixture([1.0], [[0, 100]]), [0]), [25])
for observation in (0, 150, 300):
    x, w = worked[:2]
    brute = sum(p * abs(v - observation) for v, p in zip(x, w)) - 0.5 * sum(
        p * q * abs(v - u) for v, p in zip(x, w) for u, q in zip(x, w))
    assert abs(float(mixture_crps(worked, [observation])[0]) - brute) < 1e-9
display(pd.DataFrame({"timeMs": worked[0], "probability": worked[1]}))


In [ ]:
def run_joint(mode="smoke", output=None):
    """가상 Smoke 또는 고정 Train/Validation Full을 검증하고 새 report만 게시한다."""
    require(mode in ("smoke", "full"), "INVALID_RUN_MODE")
    require(mode != "full" or output is not None, "FULL_OUTPUT_REQUIRED")
    require(definition_cells(BASELINE_NOTEBOOK, {"shared-definitions", "input-definitions", "metric-definitions"})[1]
            == BASELINE_DEFS_SHA and definition_cells(READINESS_NOTEBOOK)[1] == READINESS_DEFS_SHA,
            "DEPENDENCY_DEFINITIONS_CHANGED")
    destination = None
    if output is not None:
        output = Path(output)
        if not output.is_absolute():
            output = ROOT / output
        # 사용자가 지정한 디렉터리만 만들며 기존 leaf/alias는 공통 보호가 거부한다.
        output.parent.mkdir(parents=True, exist_ok=True)
        destination = checked_paths(ROOT, str(SOURCE_NOTEBOOK.relative_to(ROOT)), str(output), mode)[1]
    watched = {p: sha256_file(p) for p in (SOURCE_NOTEBOOK, BASELINE_NOTEBOOK, READINESS_NOTEBOOK)}
    if mode == "smoke":
        rows = synthetic_rows()
        provenance = {"kind": "synthetic_only"}
    else:
        manifest, rows, sources = load_inputs(
            os.environ.get("HUMAN_ACTION_TIMING_JOINT_DATASET", baseline["DEFAULT_DATASET"]), "full")
        watched.update(sources)
        for split, (games, count, digest) in EXPECTED_FILES.items():
            require(len(rows[split]) == count and len({r["gameId"] for r in rows[split]}) == games
                    and manifest["files"][split + ".jsonl"]["sha256"] == digest, "FIXED_COHORT_MISMATCH")
        provenance = {"kind": "fixed_legacy_train_validation",
                      "manifestSha256": baseline["EXPECTED_MANIFEST_SHA"],
                      "files": {s: manifest["files"][s + ".jsonl"] for s in EXPECTED_FILES}}
    for split in ("train", "validation"):
        validate_joint_rows(rows[split], split)
    require(not {r["gameId"] for r in rows["train"]} & {r["gameId"] for r in rows["validation"]},
            "GAME_SPLIT_LEAKAGE")
    require(not {r["sampleId"] for r in rows["train"]} & {r["sampleId"] for r in rows["validation"]},
            "DUPLICATE_SAMPLE")
    bundle = fit_joint(rows["train"])
    evaluation = evaluate_joint(bundle, rows["validation"])
    report = {"reportSchemaVersion": 1, "experimentId": EXPERIMENT_ID, "runMode": mode,
        "status": "action_timing_joint_smoke_valid" if mode == "smoke" else "action_timing_joint_validation_complete",
        "fullCohortCovered": mode == "full", "policyDecisionEligible": mode == "full",
        "canonicalResultEligible": False, "productionIntegration": False,
        "testFileRead": False, "testEvaluated": False, "previousTestAlreadyObserved": True,
        "signalOnly": True, "selectionDecision": "needs_policy_decision" if mode == "full" else "not_eligible",
        "targetPopulation": "pooled_human_not_personalized",
        "inputFeatures": ["previousAction"], "labels": ["nextActionDiagnostic", "recordedIntervalMs"],
        "gameIdLeakage": 0, "modelFitCount": 2, "evaluatedModels": list(VARIANTS),
        "primary": "validation_game_macro_next_action_nll",
        "timeGuardrail": "validation_game_macro_marginal_crps_ms",
        "cohort": {s: {"games": len({r["gameId"] for r in rr}), "pairs": len(rr)} for s, rr in rows.items()},
        "candidateBundle": bundle, "candidateBundleSha256": hashlib.sha256(json_bytes(bundle)).hexdigest(),
        "validation": evaluation, "dataset": provenance, "inputUnchanged": True,
        "baselineDefinitionsSha256": BASELINE_DEFS_SHA, "readinessDefinitionsSha256": READINESS_DEFS_SHA,
        "sourceNotebookSha256": watched[SOURCE_NOTEBOOK],
        "baselineNotebookSha256": watched[BASELINE_NOTEBOOK],
        "readinessNotebookSha256": watched[READINESS_NOTEBOOK],
        "environment": {"python": platform.python_version(), "numpy": np.__version__,
                        "pandas": pd.__version__, "matplotlib": __import__("matplotlib").__version__,
                        "scikitLearn": __import__("sklearn").__version__},
        "limits": ["one_step_not_rollout", "recorded_attempt_interval_not_keypress",
                   "no_board_placement_hold_or_first_delay", "legacy_ruleset_not_current_runtime",
                   "validation_exploration_after_observed_test", "no_new_independent_human_evidence"]}
    require(all(sha256_file(p) == h for p, h in watched.items()), "INPUT_CHANGED")
    json_bytes(report)
    if destination is not None:
        publish_report(report, destination)
    return report


## 5. 내부 검사 — 실제 데이터 없이

확률 합·fallback·입력 순서·정답 누수·지표 손계산·출력 보호를 검사합니다.
이 검사는 Notebook 안에서 실행되며 별도 Python 테스트 파일은 만들지 않습니다.


In [ ]:
def self_checks():
    import copy
    import tempfile
    from unittest.mock import patch

    checks = []

    def rejects(call):
        try:
            call()
        except (ValueError, AssertionError, FileExistsError, OSError):
            return
        raise AssertionError("EXPECTED_REJECTION")

    rows = synthetic_rows()
    bundle = fit_joint(rows["train"])
    probability, values, sources, unseen = forecast(bundle, VARIANTS[1], "move_left")
    assert abs(sum(probability) - 1) < 1e-12 and np.all(probability > 0)
    assert np.allclose(probability, (np.array(bundle["byPrevious"]["move_left"]["counts"]) + 1) / 12)
    assert np.argmax(forecast(bundle, VARIANTS[0], "move_left")[0]) == 0
    assert forecast(bundle, VARIANTS[1], "soft_drop")[3] is True
    assert forecast(bundle, VARIANTS[1], "rotate")[2][ACTIONS.index("hard_drop")] == "y"
    assert sources[ACTIONS.index("soft_drop")] == "all"
    assert json_bytes(fit_joint(list(reversed(rows["train"])))) == json_bytes(bundle)
    checks.append("counts_laplace_tie_fallback_order")

    changed = copy.deepcopy(rows["validation"])
    changed[0]["recordedIntervalMs"] = 999
    changed[-1]["nextActionDiagnostic"] = "soft_drop"
    evaluate_joint(bundle, changed)
    assert json_bytes(fit_joint(rows["train"])) == json_bytes(bundle)
    assert np.array_equal(forecast(bundle, VARIANTS[1], changed[0]["previousAction"])[0], probability)
    assert json_bytes(evaluate_joint(bundle, rows["validation"])) == json_bytes(
        evaluate_joint(bundle, list(reversed(rows["validation"]))))
    checks.append("train_only_x_only_determinism")

    for bad in (True, -1, float("nan"), float("inf"), 1.5):
        invalid = copy.deepcopy(rows["train"])
        invalid[0]["recordedIntervalMs"] = bad
        rejects(lambda: validate_joint_rows(invalid, "train"))
    rejects(lambda: validate_joint_rows(rows["train"] + rows["train"][:1], "train"))
    invalid = copy.deepcopy(rows["train"])
    invalid[0]["nextActionDiagnostic"] = "hold"
    rejects(lambda: validate_joint_rows(invalid, "train"))
    invalid = copy.deepcopy(rows["train"])
    invalid[0]["currentInputIndex"] += 1
    rejects(lambda: validate_joint_rows(invalid, "train"))
    assert mixture_crps(prepare_mixture([1.0], [[50]]), [0]).tolist() == [50.0]
    rejects(lambda: prepare_mixture([1.0], [[float("nan")]]))
    checks.append("schema_lineage_zero_finite")

    # Smoke가 실제 dataset·이전 report를 읽는 경로에 도달하면 즉시 실패한다.
    with patch.dict(run_joint.__globals__, {"load_inputs": lambda *a, **kw:
                                           (_ for _ in ()).throw(AssertionError("REAL_INPUT_OPEN"))}):
        first, second = run_joint(), run_joint()
    assert json_bytes(first) == json_bytes(second)
    assert not first["fullCohortCovered"] and not first["policyDecisionEligible"]
    checks.append("synthetic_smoke_no_dataset_determinism")

    with tempfile.TemporaryDirectory() as folder:
        parent = Path(folder)
        original = parent / "existing.json"
        original.write_bytes(b"preserve")
        rejects(lambda: checked_paths(ROOT, str(SOURCE_NOTEBOOK.relative_to(ROOT)), str(original), "smoke"))
        alias = parent / "alias.json"
        alias.symlink_to(original)
        rejects(lambda: checked_paths(ROOT, str(SOURCE_NOTEBOOK.relative_to(ROOT)), str(alias), "smoke"))
        hard = parent / "hard.json"
        os.link(original, hard)
        rejects(lambda: checked_paths(ROOT, str(SOURCE_NOTEBOOK.relative_to(ROOT)), str(hard), "smoke"))
        dangling = parent / "dangling.json"
        dangling.symlink_to(parent / "absent")
        rejects(lambda: checked_paths(ROOT, str(SOURCE_NOTEBOOK.relative_to(ROOT)), str(dangling), "smoke"))
        rejects(lambda: run_joint("smoke", original.parent / "child" / ".." / "existing.json"))
        fresh = checked_paths(ROOT, str(SOURCE_NOTEBOOK.relative_to(ROOT)), str(parent / "fresh.json"), "smoke")[1]
        publish_report(first, fresh)
        rejects(lambda: publish_report(second, fresh))
        assert original.read_bytes() == b"preserve"
    checks.append("exclusive_publish_alias_symlink_hardlink")

    source = json.loads(SOURCE_NOTEBOOK.read_text(encoding="utf-8"))
    ids = [c["id"] for c in source["cells"]]
    assert len(ids) == len(set(ids))
    checks.append("source_ids")
    return pd.DataFrame({"internalCheck": checks, "result": "passed"})


display(self_checks())


In [ ]:
def show_joint(report):
    """보존된 report 값만 표/그래프로 표시한다. 재학습·외부 report 로드는 하지 않는다."""
    mode = report["runMode"]
    scope = "synthetic smoke" if mode == "smoke" else "fixed Validation"
    evaluation = report["validation"]
    summary = pd.DataFrame({v: evaluation["models"][v]["gameMacro"] for v in VARIANTS}).T
    summary.index.name = "variant"
    print(f"{scope} | games={report['cohort']['validation']['games']} | "
          f"pairs={report['cohort']['validation']['pairs']} | NLL: nats, CRPS/MAE: ms")
    display(summary)
    display(pd.DataFrame([evaluation["gameMacroDelta"]], index=["previous_action - global"]))
    display(pd.DataFrame(evaluation["perGameDelta"]).set_index("gameId"))
    support = []
    for variant in VARIANTS:
        for row in evaluation["models"][variant]["perAction"]:
            support.append({"variant": variant, **row})
    display(pd.DataFrame(support))
    display(pd.DataFrame({v: evaluation["models"][v]["fallbackCalls"] for v in VARIANTS}).fillna(0))
    print("Fallback xy/y/all counts: 6 time-CDF calls per Validation pair; unseenPreviousRows: pair count.")
    forecasts = [{"variant": v, "previousAction": x, **f["mixtureTime"]}
                 for v in VARIANTS for x, f in evaluation["models"][v]["forecastByPrevious"].items()]
    display(pd.DataFrame(forecasts))

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    for ax, metric, unit in zip(axes, ("nextActionNll", "macroF1", "marginalCrpsMs"),
                                ("NLL (nats, lower better)", "Macro F1 (higher better)", "CRPS (ms, lower better)")):
        summary[metric].plot.bar(ax=ax, rot=20)
        ax.set_ylabel(unit)
        ax.set_xlabel("variant")
    fig.suptitle(f"{scope}: game-macro; {report['cohort']['validation']['games']} games")
    plt.tight_layout()
    plt.show()
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, v in zip(axes, VARIANTS):
        matrix = evaluation["models"][v]["confusionMatrix"]
        image = ax.imshow(matrix, cmap="Blues")
        ax.set_xticks(range(6), ACTIONS, rotation=70)
        ax.set_yticks(range(6), ACTIONS)
        ax.set_xlabel("predicted action")
        ax.set_ylabel("observed action")
        ax.set_title(v)
        fig.colorbar(image, ax=ax, label="Validation pair count")
    fig.suptitle(scope)
    plt.tight_layout()
    plt.show()
    print("Analysis:", evaluation["analysisStatus"], "| decision:", report["selectionDecision"])
    print("Smoke is not performance evidence. Oracle conditional CRPS is not the time guardrail.")


## 6. Smoke — 설정 → 실행 → 결과
외부 데이터 없이 가상 Train 3게임/9 pair와 Validation 2게임/6 pair로 같은 fit/score 경로를 확인합니다.
실제 Human 성능은 이 숫자로 판단하지 않습니다.


In [ ]:
SMOKE_OUTPUT = os.environ.get("HUMAN_ACTION_TIMING_JOINT_SMOKE_REPORT") or None


In [ ]:
SMOKE_REPORT = run_joint("smoke", SMOKE_OUTPUT)
print(SMOKE_REPORT["status"])


In [ ]:
show_joint(SMOKE_REPORT)


## 7. Full — 설정 → 실행 → 결과

이 섹션만 실제 **Train 149게임/41,151 pair → Validation 31게임/10,180 pair**를 사용합니다. Test는 열지 않습니다.

1. 가상 Smoke와 내부 검사를 먼저 확인합니다.
2. 아래 설정의 `RUN_FULL`을 `True`로 바꿉니다.
3. `FULL_OUTPUT`을 **아직 존재하지 않는** JSON 경로로 정합니다.
4. Full 실행 셀과 결과 셀을 차례대로 실행합니다.

이미 결과가 있으면 덮어쓰지 않고 중단합니다. 재실행 시 `full-report-run2.json`처럼 새 이름을 지정하세요.
최초 계산 구현의 결정성을 확인하려면 별도 사용자 실행으로 run1/run2를 비교할 수 있습니다.
실제 Full은 시간이 걸릴 수 있지만 GPU·신경망 학습은 아닙니다.

Dataset 환경변수: `HUMAN_ACTION_TIMING_JOINT_DATASET` (기본 고정 dataset).
출력 환경변수: `HUMAN_ACTION_TIMING_JOINT_FULL_REPORT`.


In [ ]:
RUN_FULL = False
FULL_OUTPUT = os.environ.get("HUMAN_ACTION_TIMING_JOINT_FULL_REPORT",
    str(ROOT / "data/ml-experiments/human-action-timing-joint-baseline/full-report-run1.json"))


In [ ]:
FULL_REPORT = None
if RUN_FULL:
    FULL_REPORT = run_joint("full", FULL_OUTPUT)
    print(FULL_REPORT["status"])
else:
    print("Full skipped. RUN_FULL=True로 바꾸기 전에는 실제 dataset/Test를 열지 않습니다.")


In [ ]:
if FULL_REPORT is not None:
    show_joint(FULL_REPORT)
else:
    print("Full 결과 없음. 위 Smoke 수치를 Human Validation 성능으로 해석하지 마세요.")


## 8. 결과를 읽는 순서와 다음 gate

1. cohort·checksum·game leakage·inputUnchanged부터 확인합니다.
2. game-macro NLL 감소 여부를 봅니다. Delta는 항상 **previous-action − global**입니다.
3. marginal CRPS가 증가했는지, F1이 감소했는지 함께 봅니다.
4. 행동별 support·confusion matrix·game별 편차·fallback을 확인합니다. Validation의 rotate_ccw support=0은 희소 행동 일반화 검증이 아닙니다.
5. NLL 감소 + 시간/F1 무악화: `conditional_signal_observed`. NLL 감소 + guardrail 악화: `tradeoff_requires_review`. 그 외: `no_clear_gain`.

세 경우 모두 자동 채택하지 않습니다. NLL 차이는 행동 확률 예측 근거이지 상태 이해·실제 플레이 증거가 아닙니다.
B3와 직접 우열을 비교하거나 이미 본 Test를 새로운 독립 검증이라고 부르지 않습니다.

다음은 **사용자 Full 결과 해석**입니다. 그 뒤 별도 요청으로 기존 raw의 Human 보드·조각 → selectedPlacement 학습 계약을 준비합니다.
전체 목표는 배치와 조작·리듬 모두 모방하는 것이며, 결합 평가에서는 모델이 예측한 배치 목표를 사용합니다.
이번 Notebook에 Hold·배치 feature·신경망·runtime을 추가하지 않습니다.
